# Task2 技术指标分析 walkthrough

这份 notebook 用教学方式复盘 Task2：本地行情 CSV 如何诊断、指标如何计算、图表和网页如何生成。Notebook 展示关键路径，完整细节见 `scripts/`。

## Goal

目标是把两个本地股票 CSV 做成一个技术指标 demo：

1. 检查缺失值和描述性统计量。
2. 计算 RSI、MACD、布林带和 ATR。
3. 用 matplotlib 绘制指标图。
4. 生成 `outputs/` 和 `web/index.html`。

## Setup

环境由根目录 `uv` 项目管理：

```powershell
uv sync --group dev
```

In [1]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
TASK_DIR = ROOT / "Task2" if (ROOT / "Task2").exists() else ROOT
SCRIPTS = TASK_DIR / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from build_site import build_site
from calculate_indicators import DATA_DIR, add_indicators, load_prices, run

## Steps

### 1. 找到输入数据

Task2 不需要联网，直接使用 `data/` 中的 CSV。

In [2]:
csv_files = sorted(DATA_DIR.glob("*.csv"))
pd.DataFrame({"file": [p.name for p in csv_files], "size_bytes": [p.stat().st_size for p in csv_files]})

,file,size_bytes
0,三一重工行情数据.csv,10749
1,平安集团行情数据.csv,31041


### 2. 数据诊断

先检查缺失值和基本统计量。真实分析里，这一步比直接画图更重要。

In [3]:
frames = {p.stem.replace("行情数据", ""): load_prices(p) for p in csv_files}
missing = pd.DataFrame(
    [{"stock": name, "missing_total": int(frame.isna().sum().sum()), "rows": len(frame)} for name, frame in frames.items()]
)
missing

,stock,missing_total,rows
0,三一重工,0,129
1,平安集团,0,372


In [4]:
sample_name = next(iter(frames))
frames[sample_name][["open", "high", "low", "close", "pct_chg", "vol"]].describe()

,open,high,low,close,pct_chg,vol
count,129.000000,129.000000,129.000000,129.000000,129.000000,1.290000e+02
mean,18.244109,18.463256,18.037597,18.254884,0.127876,6.120985e+05
std,1.374028,1.377328,1.345266,1.362506,1.679533,3.408728e+05
min,15.310000,15.540000,15.230000,15.310000,-5.759200,2.223825e+05
25%,17.600000,17.760000,17.320000,17.660000,-0.812100,3.810402e+05
50%,18.450000,18.880000,18.340000,18.550000,0.000000,4.933179e+05
75%,19.200000,19.400000,18.960000,19.190000,0.905000,7.051089e+05
max,20.500000,20.750000,20.230000,20.540000,6.281700,2.198570e+06


### 3. 指标公式的最小理解

- RSI 衡量上涨和下跌动量的相对强弱。
- MACD 比较短期 EMA 和长期 EMA 的差。
- 布林带用均线和滚动标准差描述价格波动区间。
- ATR 衡量真实波动幅度，不判断方向。

In [5]:
sample = add_indicators(frames[sample_name])
sample[["trade_date", "close", "rsi_14", "macd", "macd_signal", "bb_upper_20", "bb_lower_20", "atr_14"]].tail()

,trade_date,close,rsi_14,macd,macd_signal,bb_upper_20,bb_lower_20,atr_14
124,2025-07-10,18.92,61.972255,0.108013,-0.065920,19.000287,17.156713,0.350605
125,2025-07-11,18.83,59.758181,0.130860,-0.026564,19.097547,17.163453,0.344848
126,2025-07-14,18.74,57.544164,0.140089,0.006767,19.167093,17.186907,0.338073
127,2025-07-15,18.90,60.356200,0.158487,0.037111,19.256328,17.184672,0.329639
128,2025-07-16,18.71,55.643035,0.155939,0.060876,19.303052,17.230948,0.325379


### 4. 用几行代码画一个核心图

完整图表在 `calculate_indicators.py` 中生成。这里展示“收盘价 + 布林带”的核心。

In [6]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sample["trade_date"], sample["close"], label="收盘价")
ax.plot(sample["trade_date"], sample["bb_middle_20"], label="布林带中轨")
ax.fill_between(sample["trade_date"], sample["bb_lower_20"], sample["bb_upper_20"], alpha=0.18)
ax.set_title(f"{sample_name} 收盘价与布林带")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

C:\Users\13377\AppData\Local\Temp\ipykernel_40560\3585861921.py:8: UserWarning: Glyph 19977 (\N{CJK UNIFIED IDEOGRAPH-4E09}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_40560\3585861921.py:8: UserWarning: Glyph 19968 (\N{CJK UNIFIED IDEOGRAPH-4E00}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_40560\3585861921.py:8: UserWarning: Glyph 37325 (\N{CJK UNIFIED IDEOGRAPH-91CD}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_40560\3585861921.py:8: UserWarning: Glyph 24037 (\N{CJK UNIFIED IDEOGRAPH-5DE5}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_40560\3585861921.py:8: UserWarning: Glyph 25910 (\N{CJK UNIFIED IDEOGRAPH-6536}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_40560\3585861921.py:8: UserWarning: Glyph 30424 (\N{CJK 

### 5. 一键生成交付文件

`run()` 负责指标和图表，`build_site()` 负责网页。

In [7]:
result = run()
html_path = build_site()
html_path, result["chart_paths"]

(WindowsPath('C:/Users/13377/Desktop/PKU-WorkShop-202607/Task2/web/index.html'),
 [WindowsPath('C:/Users/13377/Desktop/PKU-WorkShop-202607/Task2/outputs/600031_SH_technical_indicators.png'),
  WindowsPath('C:/Users/13377/Desktop/PKU-WorkShop-202607/Task2/outputs/000001_SZ_technical_indicators.png')])

## Checks

确认网页、指标 CSV 和图表都已经生成。

In [8]:
assert html_path.exists()
for path in result["indicator_paths"] + result["chart_paths"]:
    assert path.exists() and path.stat().st_size > 0
"Task2 walkthrough checks passed"

'Task2 walkthrough checks passed'

## Next Steps

- 想换指标时，优先改 `calculate_indicators.py`。
- 想改页面样式时，只改 `build_site.py`。
- 想扩展更多股票时，把同字段 CSV 放进 `data/` 后重新运行 `run_all.py`。